# Exp6: 基于集成学习的 Amazon 用户评论质量预测

## 一、案例简介

随着电商平台的兴起，以及疫情的持续影响，线上购物在我们的日常生活中扮演着越来越重要的角色。在进行线上商品挑选时，评论往往是我们十分关注的一个方面。然而目前电商网站的评论质量参差不齐，甚至有水军刷好评或者恶意差评的情况出现，严重影响了顾客的购物体验。因此，对于评论质量的预测成为电商平台越来越关注的话题，如果能自动对评论质量进行评估，就能根据预测结果避免展现低质量的评论。本案例中我们将基于集成学习的方法对 Amazon 现实场景中的评论质量进行预测。

## 二、作业说明

本案例中需要大家完成两种集成学习算法的实现（Bagging、AdaBoost.M1），其中基分类器要求使用 SVM 和决策树两种，因此，一共需要对比四组结果（[AUC](https://scikit-learn.org/stable/modules/model_evaluation.html#roc-metrics) 作为评价指标）：

* Bagging + SVM
* Bagging + 决策树
* AdaBoost.M1 + SVM
* AdaBoost.M1 + 决策树

注意集成学习的核心算法需要**手动进行实现**，基分类器可以调库。

### 基本要求
* 根据数据格式设计特征的表示
* 汇报不同组合下得到的 AUC
* 结合不同集成学习算法的特点分析结果之间的差异
* （使用 sklearn 等第三方库的集成学习算法会酌情扣分）

### 扩展要求
* 尝试其他基分类器（如 k-NN、朴素贝叶斯）
* 分析不同特征的影响
* 分析集成学习算法参数的影响

## 三、实验流程

### 导入工具包

In [ ]:
import os
import time
import nltk
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tabulate import tabulate
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.decomposition import TruncatedSVD
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.utils import check_random_state
from sklearn.utils.multiclass import unique_labels
from sklearn.utils.validation import check_X_y, check_is_fitted
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import StandardScaler, MaxAbsScaler, MinMaxScaler
from collections import Counter
from joblib import Parallel, delayed
from imblearn.over_sampling import RandomOverSampler
from typing import Tuple, Dict, List, Optional, Type, Any

### 环境配置

In [ ]:
current_dir = Path.cwd()
nltk_data_path = os.path.join(current_dir, "nltk_data")
nltk.data.path.append(str(nltk_data_path))

plt.rcParams["font.family"] = ["SimHei"]
plt.rcParams["axes.unicode_minus"] = False
# plt.rcParams['font.size'] = 20
RANDOM_SEED = 2025

# 配置日志记录器
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler()],
)  # 仅控制台输出
logger = logging.getLogger("Record")

### 定义相关的类

#### 数据加载类

In [ ]:
class DataLoader:
    """
    数据加载器类，负责加载和预处理数据集

    功能:
    - 加载训练数据、测试数据和真实标签数据
    - 通过Id合并测试数据与真实标签
    - 分割特征和标签
    """

    def __init__(self, train_path: str, test_path: str, ground_truth_path: str) -> None:
        """
        参数:
            train_path (str): 训练数据文件路径
            test_path (str): 测试数据文件路径
            ground_truth_path (str): 测试标签文件路径
        """

        self.train_path = train_path
        self.test_path = test_path
        self.ground_truth_path = ground_truth_path

    def load_data(self) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series]:
        """
        加载并预处理数据

        步骤:
        1. 加载训练数据
        2. 加载测试数据
        3. 加载真实标签数据
        4. 通过Id将真实标签合并到测试数据中
        5. 分割特征和标签

        返回:
            Tuple: (X_train, y_train, X_test, y_test)
                X_train (pd.DataFrame): 训练特征
                y_train (pd.Series): 训练标签
                X_test (pd.DataFrame): 测试特征
                y_test (pd.Series): 测试标签
        """

        start_time = time.time()
        logger.info("开始加载数据集...")

        try:
            # 加载训练数据
            logger.info(f"加载训练数据: {self.train_path}")
            train_df = pd.read_csv(self.train_path, sep="\t")

            # 加载测试数据
            logger.info(f"加载测试数据: {self.test_path}")
            test_df = pd.read_csv(self.test_path, sep="\t")

            # 加载真实标签
            logger.info(f"加载真实标签: {self.ground_truth_path}")
            ground_truth_df = pd.read_csv(self.ground_truth_path, sep=",")

            # 通过Id合并测试数据与真实标签
            logger.info("通过Id合并测试数据与真实标签")
            test_df = test_df.merge(ground_truth_df, on="Id", how="left")

            # 提取特征和标签
            logger.info("分割特征和标签")
            X_train_temp = train_df[["reviewText"]]  # 仅选择英文评论文本内容作为特征
            y_train_temp = train_df["label"]
            X_test = test_df[["reviewText"]]  # 仅选择英文评论文本内容作为特征
            y_test = test_df["Expected"]

            logger.info(f"原始训练数据标签分布: {Counter(y_train_temp)}")

            ros = RandomOverSampler(random_state=RANDOM_SEED)
            X_train, y_train = ros.fit_resample(X_train_temp, y_train_temp)

            logger.info(f"过采样后训练数据标签分布: {Counter(y_train)}")

            elapsed = time.time() - start_time
            logger.info(f"数据集加载完成! 耗时: {elapsed:.2f}秒")
            logger.info(f"训练集形状: {X_train.shape}, 测试集形状: {X_test.shape}")

            return X_train, y_train, X_test, y_test

        except Exception as e:
            logger.error(f"加载数据时出错: {str(e)}")
            raise

#### 特征工程类

In [ ]:
class FeatureEngineer:
    """
    特征工程类，负责特征提取和转换

    功能:
    - 使用NLTK进行预处理文本特征（TF-IDF向量化）
    - 支持TruncatedSVD降维
    - 提供静态方法进行文本预处理
    """

    def __init__(
        self,
        text_column: str = "reviewText",
        vectorizer_type: str = "tfidf",
        scaling_method: str = "zscore",
        max_features: int = 50000,
        n_components: int = 50,
    ) -> None:
        """
        参数:
                text_column (str): 文本列名，默认为'reviewText'
                vectorizer_type (str): 词频向量化方式，默认'tfidf'，可选'count'/'tfidf'
                max_features (int): TF-IDF最大特征数，默认为10000
                n_components (int): 对向量化的文本特征进行降维，默认为50，可选None
        """

        self.text_column = text_column
        self.max_features = max_features
        self.vectorizer = None  # TF-IDF向量化器
        self.scaler = StandardScaler()  # 标准化器
        self.cat_encoders = {}  # 分类特征编码器
        self.vectorizer_type = vectorizer_type
        self.n_components = n_components
        self.scaling_method = scaling_method
        self.scaler = None

        # 初始化降维组件（仅当需要降维时）
        if self.n_components and self.n_components > 0:
            self.svd = TruncatedSVD(n_components=n_components)
        else:
            self.svd = None

        # 初始化NLTK组件
        self.stop_words = set(stopwords.words("english"))
        self.lemmatizer = WordNetLemmatizer()

    def _preprocess_text(self, text: str) -> str:
        """
        使用NLTK进行文本预处理:
        1. 分词
        2. 转换为小写
        3. 移除标点符号和停用词
        4. 词形还原

        参数:
                text (str): 输入文本

        返回:
                str: 预处理后的文本
        """

        if not isinstance(text, str) or text.strip() == "":
            return ""

        tokens = word_tokenize(text.lower())  # 分词和小写化
        tokens = [
            token for token in tokens if token.isalpha()
        ]  # 移除标点符号和非字母字符
        tokens = [
            token for token in tokens if token not in self.stop_words
        ]  # 移除停用词
        tokens = [self.lemmatizer.lemmatize(token) for token in tokens]  # 词形还原

        return " ".join(tokens)

    def fit_transform(self, X: pd.DataFrame) -> np.ndarray:
        """
        拟合并转换特征

        步骤:
        1. 使用NLTK预处理文本特征
        2. 使用向量化器对文本进行处理
        3. 可选降维处理
        4. 返回处理后的特征

        参数:
                X (pd.DataFrame): 输入数据

        返回:
                np.ndarray: 转换后的特征矩阵
        """

        start_time = time.time()
        logger.info("开始特征工程 - fit_transform...")

        try:
            # 步骤1：NLTK预处理
            text_data = X[self.text_column].fillna("").apply(self._preprocess_text)
            logger.info(f"完成NLTK预处理，样本数: {len(text_data)}")

            # 步骤2：文本向量化
            if self.vectorizer_type == "count":
                self.vectorizer = CountVectorizer(
                    max_features=self.max_features,
                    ngram_range=(1, 3),
                    min_df=3,
                    max_df=0.95,
                    stop_words="english",
                )
            else:
                self.vectorizer = TfidfVectorizer(
                    max_features=self.max_features,
                    ngram_range=(1, 3),
                    min_df=3,
                    max_df=0.95,
                    stop_words="english",
                )

            # 生成向量化特征
            tfidf_features = self.vectorizer.fit_transform(text_data)
            logger.info(f"TF-IDF特征生成完成，形状: {tfidf_features.shape}")

            # 步骤3：降维处理
            if self.svd:
                reduced_features = self.svd.fit_transform(tfidf_features)
                logger.info(
                    f"TruncatedSVD降维完成，从{tfidf_features.shape[1]}→{self.n_components}维"
                )
            else:
                reduced_features = tfidf_features.toarray()  # 若不降维则转换为密集矩阵
                logger.info("未启用降维，直接使用文本向量化特征")

            if self.scaling_method and reduced_features.shape[0] > 0:
                # logger.info(f"应用特征标准化: {self.scaling_method}")
                if self.scaling_method == "zscore":
                    self.scaler = StandardScaler()
                    scaled_features = self.scaler.fit_transform(reduced_features)
                elif self.scaling_method == "minmax":
                    self.scaler = MinMaxScaler()
                    scaled_features = self.scaler.fit_transform(reduced_features)
                elif self.scaling_method == "maxabs":
                    self.scaler = MaxAbsScaler()
                    scaled_features = self.scaler.fit_transform(reduced_features)
                else:
                    scaled_features = reduced_features
            else:
                scaled_features = reduced_features

            elapsed = time.time() - start_time
            logger.info(
                f"特征工程完成! 最终特征形状: {reduced_features.shape}, 耗时: {elapsed:.2f}秒"
            )
            return scaled_features

        except Exception as e:
            logger.error(f"特征工程处理出错: {str(e)}", exc_info=True)
            raise

    def transform(self, X: pd.DataFrame) -> np.ndarray:
        """
        使用预先拟合的转换器转换特征

        步骤:
        1. 使用NLTK预处理应用文本特征转换
        2. 使用向量化器对文本进行处理
        3. 可选降维处理
        4. 返回处理后的特征

        参数:
                X (pd.DataFrame): 输入数据

        返回:
                np.ndarray: 转换后的特征矩阵
        """

        start_time = time.time()
        logger.info("开始特征工程 - transform...")

        try:
            if not self.vectorizer:
                raise RuntimeError("向量化器尚未拟合，请先调用fit_transform方法")

            # 步骤1：NLTK预处理（与训练集一致）
            text_data = X[self.text_column].fillna("").apply(self._preprocess_text)

            # 步骤2：TF-IDF转换（使用训练集拟合的vectorizer）
            tfidf_features = self.vectorizer.transform(text_data)
            logger.info(f"TF-IDF转换完成，形状: {tfidf_features.shape}")

            # 步骤3：降维转换
            if self.svd:
                reduced_features = self.svd.transform(tfidf_features)
                logger.info(
                    f"TruncatedSVD降维转换完成，从{tfidf_features.shape[1]}→{self.n_components}维"
                )
            else:
                reduced_features = tfidf_features.toarray()  # 转换为密集矩阵
                logger.info("未启用降维，直接使用文本向量化特征")

            if self.scaler and self.scaling_method:
                scaled_features = self.scaler.transform(reduced_features)
                # logger.info(f"应用{self.scaling_method}标准化转换")
            else:
                scaled_features = reduced_features

            elapsed = time.time() - start_time
            logger.info(
                f"特征转换完成! 最终特征形状: {reduced_features.shape}, 耗时: {elapsed:.2f}秒"
            )
            return scaled_features

        except Exception as e:
            logger.error(f"特征转换出错: {str(e)}", exc_info=True)
            raise

    @staticmethod
    def preprocess_text_only(text_series: pd.Series) -> pd.Series:
        """
        只进行文本预处理，返回处理后的文本序列
        避免在特征分析中重复预处理
        此方法不依赖于类实例，可直接调用

        参数:
                text_series (pd.Series): 需要处理的文本特征

        返回:
                pd.Series: 处理后的文本序列，长度与输入一致，每个元素为经过标准化的文本字符串
        """

        lemmatizer = WordNetLemmatizer()
        stop_words = set(stopwords.words("english"))

        def _preprocess(text: str) -> str:
            if not isinstance(text, str) or text.strip() == "":
                return ""
            tokens = word_tokenize(text.lower())
            tokens = [token for token in tokens if token.isalpha()]
            tokens = [token for token in tokens if token not in stop_words]
            tokens = [lemmatizer.lemmatize(token) for token in tokens]
            return " ".join(tokens)

        return text_series.fillna("").apply(_preprocess)

#### 基分类器包装器类

In [ ]:
class BaseClassifierWrapper(BaseEstimator, ClassifierMixin):
    """
    基分类器包装器，提供统一的分类器接口

    功能:
    - 统一不同分类器的接口
    - 处理样本权重支持
    - 将决策函数转换为概率（对于不支持概率的模型）
    """

    def __init__(
        self, base_estimator: BaseEstimator, supports_weights: bool = False
    ) -> None:
        """
        参数:
                base_estimator: 基分类器对象，必须是BaseEstimator的实例
                supports_weights (bool): 是否支持样本权重，默认为False
        """

        super().__init__()

        if not isinstance(
            base_estimator, BaseEstimator
        ):  # 验证base_estimator是有效的估计器
            raise ValueError("base_estimator必须是sklearn的BaseEstimator实例")

        self.base_estimator = base_estimator
        self.supports_weights = supports_weights
        self.is_calibrated_ = False
        # logger.info(f"创建分类器包装器: {type(base_estimator).__name__}, "
        # f"支持权重: {supports_weights}")

    def get_params(self, deep: bool = True) -> Dict[str, Any]:
        """
        获取分类器参数以支持克隆和超参数调优

        参数:
                deep (bool): 是否递归获取嵌套参数，默认为True
        """

        # 获取基类参数
        params = super().get_params(deep=False)

        # 添加当前类的参数
        params.update(
            {
                "base_estimator": self.base_estimator,
                "supports_weights": self.supports_weights,
            }
        )

        # 深度获取嵌套估计器的参数
        if deep:
            base_params = self.base_estimator.get_params(deep=True)
            for key, value in base_params.items():
                params[f"base_estimator__{key}"] = (
                    value  # 添加嵌套参数格式：base_estimator__<sub_param>
                )

        return params

    def set_params(self, **params) -> "BaseClassifierWrapper":
        """
        设置分类器参数以支持克隆和超参数调优

        参数:
                ** params: 键值对形式的参数
        """

        # 分离基础估计器参数
        base_estimator_params = {}
        for key in list(params.keys()):
            if key.startswith("base_estimator__"):
                # 转换 key：base_estimator__param -> param
                base_estimator_params[key[16:]] = params.pop(key)
            else:
                setattr(self, key, params.pop(key))

        # 设置基础估计器参数
        if base_estimator_params and hasattr(self.base_estimator, "set_params"):
            self.base_estimator.set_params(**base_estimator_params)

        return self

    def fit(
        self, X: np.ndarray, y: np.ndarray, sample_weight: Optional[np.ndarray] = None
    ) -> "BaseClassifierWrapper":
        """
        训练模型

        参数:
                X (np.ndarray): 特征矩阵，形状为(n_samples, n_features)
                y (np.ndarray): 标签向量，形状为(n_samples,)
                sample_weight (np.ndarray, 可选): 样本权重向量，形状为(n_samples,)

        返回:
                self: 训练好的分类器实例
        """

        # 输入验证
        if X.shape[0] != y.shape[0]:
            raise ValueError(
                f"特征矩阵和标签向量的样本数不匹配: {X.shape[0]} vs {y.shape[0]}"
            )

        if sample_weight is not None and X.shape[0] != sample_weight.shape[0]:
            raise ValueError(
                f"样本数和权重数不匹配: {X.shape[0]} vs {sample_weight.shape[0]}"
            )

        start_time = time.time()
        clf_name = type(self.base_estimator).__name__
        # logger.info(f"训练 {clf_name} 分类器... 样本数: {X.shape[0]}, 特征数: {X.shape[1]}")

        try:
            # 检查是否需要概率校准
            requires_calibration = not hasattr(self.base_estimator, "predict_proba")
            self.is_calibrated_ = requires_calibration

            if requires_calibration:
                # logger.info(f"{clf_name} 不支持概率预测，添加概率校准层")

                # 训练基础分类器
                if self.supports_weights and sample_weight is not None:
                    # logger.info("使用样本权重训练基础模型")
                    self.base_estimator.fit(X, y, sample_weight=sample_weight)
                else:
                    # logger.info("训练基础模型(无样本权重)")
                    self.base_estimator.fit(X, y)

                # 根据类别数量选择校准方法
                n_classes = len(np.unique(y))
                method = "sigmoid" if n_classes == 2 else "isotonic"
                # logger.info(f"使用 {method} 方法进行概率校准，类别数: {n_classes}")

                self.calibrator_ = CalibratedClassifierCV(
                    estimator=clone(self.base_estimator),  # 使用克隆避免修改原始模型
                    method=method,
                    cv=5,
                    ensemble=True,
                )
                self.calibrator_.fit(X, y)

            else:
                # 直接训练支持概率的模型
                if self.supports_weights and sample_weight is not None:
                    # logger.info("使用样本权重训练模型")
                    self.base_estimator.fit(X, y, sample_weight=sample_weight)
                else:
                    # logger.info("训练模型(无样本权重)")
                    self.base_estimator.fit(X, y)

            elapsed = time.time() - start_time
            logger.info(f"基分类器 {clf_name} 训练完成! 耗时: {elapsed:.2f}秒")
            return self

        except Exception as e:
            logger.error(f"训练 {clf_name} 出错: {str(e)}", exc_info=True)
            raise

    def predict(self, X: np.ndarray) -> np.ndarray:
        """
        预测标签

        参数:
                X (np.ndarray): 特征矩阵，形状为(n_samples, n_features)

        返回:
                np.ndarray: 预测标签，形状为(n_samples,)
        """

        clf_name = type(self.base_estimator).__name__
        # logger.info(f"使用 {clf_name} 预测标签...")

        if self.is_calibrated_:
            return self.calibrator_.predict(X)
        else:
            return self.base_estimator.predict(X)

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """
        预测概率

        参数:
                X (np.ndarray): 特征矩阵，形状为(n_samples, n_features)

        返回:
                np.ndarray: 预测概率，形状为(n_samples, n_classes)
        """

        clf_name = type(self.base_estimator).__name__
        # logger.info(f"使用 {clf_name} 预测概率...")

        if self.is_calibrated_:
            return self.calibrator_.predict_proba(X)
        else:
            return self.base_estimator.predict_proba(X)

    def decision_function(self, X: np.ndarray) -> np.ndarray:
        """
        决策函数，返回样本的原始分数

        参数:
                X (np.ndarray): 特征矩阵，形状为(n_samples, n_features)

        返回:
                np.ndarray: 决策分数，形状取决于基础分类器
        """

        clf_name = type(self.base_estimator).__name__
        logger.info(f"使用 {clf_name} 计算决策函数...")

        if self.is_calibrated_:
            return self.calibrator_.decision_function(X)
        elif hasattr(self.base_estimator, "decision_function"):
            return self.base_estimator.decision_function(X)
        else:
            raise NotImplementedError(f"{clf_name} 不支持决策函数")

#### Bagging集成学习算法类

In [ ]:
class BaggingClassifier(BaseEstimator, ClassifierMixin):
    """
    支持并行训练的Bagging集成学习算法实现

    功能:
    - 使用独立的随机状态生成器
    - 为每个基分类器分配唯一种子
    - 支持嵌套参数设置
    - 随机采样
    """

    def __init__(
        self,
        base_estimator: BaseEstimator,
        n_estimators: int = 50,
        max_samples: float = 1.0,
        n_jobs: int = 1,
        random_state: Optional[int] = RANDOM_SEED,
    ) -> None:
        """
        初始化Bagging分类器

        参数:
                base_estimator: 基分类器实例，将被集成
                n_estimators: 集成的基分类器数量，默认为50
                max_samples: 用于训练每个基分类器的样本比例，范围(0, 1]，默认1.0
                n_jobs: 并行训练的进程数，-1表示使用所有可用CPU，默认为1
                random_state: 随机数种子，用于保证结果可复现
        """

        super().__init__()
        # 验证max_samples参数有效性
        if not (0 < max_samples <= 1.0):
            raise ValueError("max_samples must be in (0, 1]")
        self.base_estimator = base_estimator
        self.n_estimators = n_estimators
        self.max_samples = max_samples
        self.n_jobs = n_jobs
        self.random_state = random_state
        self.estimators_ = []  # 存储训练好的基分类器
        self._rng = np.random.default_rng(random_state)  # 主随机数生成器

        logger.info(
            f"创建Bagging分类器: n_estimators={n_estimators}, "
            f"max_samples={max_samples}, n_jobs={n_jobs}"
        )

    def get_params(self, deep: bool = True) -> Dict[str, Any]:
        """
        获取模型参数，用于网格搜索等超参数优化

        参数:
                deep: 是否递归获取嵌套参数

        返回:
                包含模型所有参数的字典
        """

        # 获取父类参数
        params = super().get_params(deep=False)
        # 添加当前类参数
        params.update(
            {
                "n_estimators": self.n_estimators,
                "max_samples": self.max_samples,
                "n_jobs": self.n_jobs,
                "random_state": self.random_state,
            }
        )

        # 如果需要深度获取参数，递归获取基分类器参数
        if deep:
            base_params = self.base_estimator.get_params(deep=True)
            for key, value in base_params.items():
                # 以"base_estimator__参数名"格式存储基分类器参数
                params[f"base_estimator__{key}"] = value

        return params

    def set_params(self, **params) -> "BaggingClassifier":
        """
        设置模型参数，支持嵌套参数设置

        参数:
                ** params: 键值对形式的参数

        返回:
                设置好参数的当前实例
        """

        # 分离嵌套参数（基分类器参数）
        nested_params = {}
        for key, value in params.items():
            if key.startswith("base_estimator__"):
                nested_params[key] = value
            else:
                setattr(self, key, value)  # 设置当前类的参数

        # 设置基分类器参数
        if nested_params:
            # 去除参数名中的"base_estimator__"前缀
            base_params = {k.split("__", 1)[1]: v for k, v in nested_params.items()}
            self.base_estimator.set_params(**base_params)

        return self

    def _generate_bootstrap(
        self, rng: np.random.Generator, n_samples: int
    ) -> np.ndarray:
        """
        生成进程安全的自助采样索引

        参数:
                rng: 随机数生成器实例
                n_samples: 原始样本数量

        返回:
                自助采样的索引数组
        """

        # 生成指定数量的随机索引，有放回采样
        return rng.choice(
            n_samples, size=int(n_samples * self.max_samples), replace=True
        )

    def _train_estimator(
        self, estimator: BaseEstimator, X: np.ndarray, y: np.ndarray, seed: int
    ) -> BaseEstimator:
        """
        训练单个基分类器

        参数:
                estimator: 基分类器实例
                X: 特征数据
                y: 标签数据
                seed: 用于当前分类器的随机种子

        返回:
                训练好的基分类器
        """

        # 创建进程局部的随机数生成器，避免多进程间随机状态冲突
        local_rng = np.random.default_rng(seed)

        # 生成当前分类器的自助样本索引
        n_samples = X.shape[0]
        indices = self._generate_bootstrap(local_rng, n_samples)

        # 如果基分类器支持random_state参数，设置其随机种子
        estimator = clone(self.base_estimator)
        if hasattr(estimator, "set_params"):
            try:
                estimator.set_params(**{"base_estimator__random_state": seed})
            except Exception as e:
                logger.warning(f"设置随机状态失败: {str(e)}")

        # 使用自助样本训练基分类器并返回
        return estimator.fit(X[indices], y[indices])

    def fit(self, X: np.ndarray, y: np.ndarray) -> "BaggingClassifier":
        """
        训练Bagging集成分类器

        参数:
                X: 特征数据，形状为(n_samples, n_features)
                y: 标签数据，形状为(n_samples,)

        返回:
                训练好的Bagging分类器实例
        """

        start_time = time.time()
        base_name = type(self.base_estimator).__name__

        logger.info(
            f"开始训练Bagging集成({base_name})... "
            f"基分类器数: {self.n_estimators}, 样本数: {X.shape[0]}"
        )

        # 为每个基分类器生成唯一的随机种子，保证可复现性
        seeds = self._rng.integers(0, np.iinfo(np.int32).max, self.n_estimators)

        # 克隆基分类器，确保每个基分类器独立
        estimators = [clone(self.base_estimator) for _ in range(self.n_estimators)]

        # 并行训练所有基分类器
        # logger.info(f"并行训练基分类器 (n_jobs={self.n_jobs})...")
        self.estimators_ = Parallel(n_jobs=self.n_jobs, verbose=1)(
            delayed(self._train_estimator)(est, X, y, seed)
            for est, seed in zip(estimators, seeds)
        )

        clf_name = type(self.base_estimator).__name__
        logger.info(
            f"BaggingClassifier {clf_name} 训练完成! 耗时: {time.time() - start_time:.2f}秒"
        )

        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """
        预测样本属于各个类别的概率

        参数:
                X: 特征数据，形状为(n_samples, n_features)

        返回:
                概率数组，形状为(n_samples, n_classes)，每个元素表示样本属于对应类别的概率
        """

        # 并行获取每个基分类器的预测概率
        probas = Parallel(n_jobs=self.n_jobs)(
            delayed(estimator.predict_proba)(X) for estimator in self.estimators_
        )

        # 对所有基分类器的预测概率取平均
        return np.mean(probas, axis=0)

    def predict(self, X: np.ndarray) -> np.ndarray:
        """
        预测样本类别

        参数:
                X: 特征数据，形状为(n_samples, n_features)

        返回:
                预测类别数组，形状为(n_samples,)
        """

        # 基于预测概率，取概率最大的类别作为预测结果
        return np.argmax(self.predict_proba(X), axis=1)

#### AdaBoost.M1集成学习算法类

In [ ]:
class AdaBoostM1Classifier(BaseEstimator, ClassifierMixin):
    """
    AdaBoost.M1集成学习算法分类器

    AdaBoost.M1算法：
    1. 初始化样本权重为均匀分布
    2. 迭代训练多个基分类器
    3. 考虑样本权重
    4. 基于错误率计算分类器权重
    5. 组合多个弱分类器形成强分类器
    """

    def __init__(
        self,
        base_estimator: BaseEstimator,
        n_estimators: int = 50,
        random_state: Optional[int] = RANDOM_SEED,
    ) -> None:
        """
        初始化AdaBoostM1分类器

        参数:
                base_estimator: 基分类器实例
                n_estimators: 最大基分类器数量，默认为50
                random_state: 随机种子
        """

        super().__init__()
        self.base_estimator = base_estimator
        self.n_estimators = n_estimators
        self.random_state = random_state
        logger.info(f"创建AdaBoostM1分类器: n_estimators={n_estimators}")

    def get_params(self, deep: bool = True) -> Dict[str, Any]:
        """
        获取模型参数

        支持深度获取基分类器的参数

        参数:
                deep: 是否递归获取嵌套参数

        返回:
                参数字典
        """

        params = super().get_params(deep=deep)
        # 添加当前对象参数
        params.update(
            {
                "base_estimator": self.base_estimator,
                "n_estimators": self.n_estimators,
                "random_state": self.random_state,
            }
        )

        # 深度获取时，递归添加基分类器参数
        if deep and hasattr(self.base_estimator, "get_params"):
            base_params = self.base_estimator.get_params(deep=deep)
            # 添加前缀区分嵌套参数
            params.update({f"base_estimator__{k}": v for k, v in base_params.items()})

        return params

    def set_params(self, **params) -> "AdaBoostM1Classifier":
        """
        设置模型参数

        支持嵌套参数设置，用于网格搜索等场景

        参数:
                **params: 要设置的参数字典

        返回:
                更新后的自身实例
        """

        # 提取基分类器嵌套参数
        base_estimator_params = {}
        for key in list(params.keys()):
            if key.startswith("base_estimator__"):
                base_param = key.split("__", 1)[1]
                base_estimator_params[base_param] = params.pop(key)

        # 设置基分类器嵌套参数
        if base_estimator_params and hasattr(self.base_estimator, "set_params"):
            self.base_estimator.set_params(**base_estimator_params)

        # 设置当前对象参数
        super().set_params(**params)

        return self

    def fit(self, X: np.ndarray, y: np.ndarray) -> "AdaBoostM1Classifier":
        """
        训练AdaBoost.M1模型

        算法步骤:
        1. 初始化样本权重为均匀分布
        2. 迭代训练多个基分类器
        3. 通过重采样实现样本加权
        4. 计算加权错误率ε_t
        5. 加权错误率>0.5则推出循环
        6. 计算β_t = ε_t/(1-ε_t)
        7. 更新样本权重
        8. 计算分类器权重log(1/β_t)

        参数:
                X: 训练特征矩阵 (n_samples, n_features)
                y: 训练标签向量 (n_samples,)

        返回:
                训练好的实例
        """

        # 验证输入数据和标签格式
        X, y = check_X_y(X, y)
        # 记录数据集中的唯一类别标签
        self.classes_ = unique_labels(y)

        # 记录训练开始时间和基本信息
        start_time = time.time()
        base_name = type(self.base_estimator).__name__
        logger.info(
            f"开始训练AdaBoost集成({base_name})... "
            f"基分类器数: {self.n_estimators}, 训练样本数: {X.shape[0]}"
        )

        # 初始化随机数生成器
        rng = check_random_state(self.random_state)
        n_samples = X.shape[0]

        # 步骤1: 初始化样本权重 (均匀分布1/N)
        sample_weights = np.ones(n_samples) / n_samples
        logger.info(
            f"初始样本权重: 总和={np.sum(sample_weights):.4f}, 均值={np.mean(sample_weights):.4f}"
        )

        # 存储训练好的基分类器和对应权重
        self.estimators_ = []
        self.estimator_weights_ = []

        # 步骤2: 迭代训练T个基分类器
        for i in range(self.n_estimators):
            iter_start = time.time()
            # logger.info(f"训练基分类器 {i+1}/{self.n_estimators}...")

            # 使用样本权重训练基分类器，克隆基分类器以保持独立性
            estimator = clone(self.base_estimator)

            try:
                # 在加权样本上训练基分类器
                estimator.fit(X, y, sample_weight=sample_weights)
            except ValueError:
                # 显式重采样
                indices = rng.choice(
                    n_samples, size=n_samples, p=sample_weights, replace=True
                )
                estimator.fit(X[indices], y[indices])

            # 步骤3: 计算加权错误率ε_t (所有错误分类样本的权重和)
            y_pred = estimator.predict(X)
            incorrect = y_pred != y  # 标识错误分类样本
            error = np.sum(sample_weights[incorrect])  # 计算加权加权错误率
            pred_acc = accuracy_score(y_pred, y)

            logger.info(
                f"基分类器准确率acc: {pred_acc:.4f}, 分类器加权错误率ε_t: {error:.4f}"
            )

            # 步骤4: 如果加权错误率>0.5则退出循环
            if error > 0.5:
                # logger.warning(f"加权错误率ε_t={error:.4f}>0.5，退出循环")
                break  # 退出整个迭代循环

            # 步骤5: 计算β_t = ε_t/(1-ε_t)
            beta = error / (1 - error + 1e-20)  # 添加小常数避免除零错误
            # logger.info(f"计算β_t: {beta:.4f} = {error:.4f}/(1-{error:.4f})")

            # 步骤6: 更新样本权重 (核心步骤)
            # 正确分类样本: W_new = W_old * β_t (降低权重)
            # 错误分类样本: W_new = W_old (保持权重不变)
            updated_weights = sample_weights * np.where(incorrect, 1, beta)
            protected_weights = np.clip(updated_weights, 1e-30, None)
            sample_weights = protected_weights / np.sum(protected_weights)

            # logger.info(f"更新后样本权重: 总和={np.sum(sample_weights):.4f}, "
            # f"最大={np.max(sample_weights):.4f}, 最小={np.min(sample_weights):.4f}")

            # 步骤8: 保存分类器并计算投票权重
            # 投票权重 = log(1/β_t)
            alpha_val = np.log(1.0 / beta)
            self.estimators_.append(estimator)
            self.estimator_weights_.append(alpha_val)
            # logger.info(f"基分类器投票权重: log(1/β_t) = {alpha_val:.4f}")

            # 记录迭代时间
            iter_time = time.time() - iter_start
            logger.info(
                f"基分类器 {len(self.estimators_)} 训练完成! 耗时: {iter_time:.2f}秒"
            )

        # 记录总体训练信息
        total_time = time.time() - start_time
        logger.info(
            f"AdaBoost集成训练完成! 有效分类器数: {len(self.estimators_)}, 总耗时: {total_time:.2f}秒"
        )
        return self  # 返回训练好的模型

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """
        计算加权概率分布 (AdaBoost.M1算法)

        实现步骤:
        1. 验证模型是否已训练
        2. 初始化样本投票得分矩阵 (n_samples, n_classes)
        3. 遍历所有基分类器进行加权投票
                - 每个分类器预测样本类别
                - 在预测类别的投票上添加权重 log(1/β_t)
        4. 计算归一化概率分布

        参数:
                X: 待预测样本矩阵 (n_samples, n_features)

        返回:
                类别概率矩阵 (n_samples, n_classes)
        """

        # 步骤1: 验证模型是否已训练
        check_is_fitted(self, ["estimators_", "estimator_weights_", "classes_"])

        # 检查是否存在有效分类器
        if len(self.estimators_) == 0:
            logger.warning("AdaBoost模型未训练任何基分类器，返回均匀概率分布")
            n_classes = len(self.classes_)
            return np.ones((X.shape[0], n_classes)) / n_classes

        n_samples = X.shape[0]
        n_classes = len(self.classes_)

        # 步骤2: 初始化投票得分矩阵
        vote_scores = np.zeros((n_samples, n_classes))

        # 步骤3: 遍历基分类器进行加权投票
        for estimator, alpha in zip(self.estimators_, self.estimator_weights_):
            # 获取当前分类器的预测标签
            y_pred = estimator.predict(X)

            # 将预测标签映射到类别索引
            class_indices = np.array(
                [np.where(self.classes_ == label)[0][0] for label in y_pred]
            )

            # 使用高级索引高效赋值
            sample_indices = np.arange(n_samples)
            vote_scores[sample_indices, class_indices] += alpha

        # 1.每行减去最大值
        max_scores = np.max(vote_scores, axis=1, keepdims=True)
        stable_scores = vote_scores - max_scores

        # 2.计算指数
        exp_scores = np.exp(stable_scores)

        # 3.归一化（确保每行和为1）
        row_sums = np.sum(exp_scores, axis=1, keepdims=True)
        probas = exp_scores / np.maximum(row_sums, 1e-15)

        # 4.数值裁剪（确保概率在[0,1]范围内）
        probas = np.clip(probas, 0.0, 1.0)

        # 添加概率分布的说明性输出
        logger.info(f"预测概率分布: 样本数={n_samples}, 类别数={n_classes}")
        logger.debug(
            f"投票得分范围: {np.min(vote_scores):.2f}-{np.max(vote_scores):.2f}"
        )
        logger.debug(f"概率范围: {np.min(probas):.4f}-{np.max(probas):.4f}")

        return probas

    def predict(self, X: np.ndarray) -> np.ndarray:
        """
        预测类别标签 (AdaBoost.M1加权投票算法)

        实现步骤:
        1. 验证模型是否已训练
        2. 初始化样本投票得分矩阵 (n_samples, n_classes)
        3. 遍历所有基分类器，计算加权投票:
                - 每个分类器预测样本类别
                - 在预测类别的投票上加上权重 log(1/β_t)
        4. 选择每个样本的最高得分类别作为预测结果

        参数:
                X: 待预测样本矩阵 (n_samples, n_features)

        返回:
                预测标签向量 (n_samples,)
        """

        # 步骤1: 验证模型是否已训练
        check_is_fitted(self, ["estimators_", "estimator_weights_", "classes_"])

        # 检查是否存在有效分类器
        if len(self.estimators_) == 0:
            logger.warning("AdaBoost模型未训练任何基分类器，返回随机预测")
            return np.random.choice(self.classes_, X.shape[0])

        logger.info(
            f"执行AdaBoost.M1预测... 测试样本数: {X.shape[0]}, 基分类器数: {len(self.estimators_)}"
        )

        # 步骤2: 初始化样本投票得分矩阵
        n_samples = X.shape[0]
        n_classes = len(self.classes_)
        # 创建全零投票得分矩阵 (每个样本在不同类别上的得分)
        vote_scores = np.zeros((n_samples, n_classes))

        # 步骤3: 遍历基分类器进行加权投票
        for i, (estimator, alpha) in enumerate(
            zip(self.estimators_, self.estimator_weights_)
        ):
            logger.debug(f"基分类器 {i + 1} 投票权重: {alpha:.4f}")

            # 获取当前分类器的预测标签
            y_pred = estimator.predict(X)

            # 将预测标签转换为类别索引 (用于投票)
            # 创建索引数组: [0, 1, 2, ..., n_samples-1]
            sample_indices = np.arange(n_samples)
            # 将标签映射到类别索引 (使用self.classes_的索引值)
            class_indices = np.searchsorted(self.classes_, y_pred)

            # 加权投票: 在预测类别位置上增加权重 (log(1/β_t))
            vote_scores[sample_indices, class_indices] += alpha

        # 步骤4: 选择最高得分的类别
        # 获取每个样本最高得分对应的类别索引
        winning_indices = np.argmax(vote_scores, axis=1)
        # 映射回原始类别标签
        predictions = self.classes_[winning_indices]

        # 输出预测结果统计
        unique_preds, counts = np.unique(predictions, return_counts=True)
        logger.info(f"预测结果分布: {dict(zip(unique_preds, counts))}")

        return predictions

#### 评论质量预测系统类

In [ ]:
class CommentQualityPredictor:
    """
    Amazon评论质量预测系统

    功能:
    - 训练和评估不同集成方法的组合
    - 分析特征对模型性能的影响
    - 分析参数对模型性能的影响
    - 输出性能结果
    """

    def __init__(
        self,
        base_classifiers: Optional[Dict[str, BaseClassifierWrapper]] = None,
        ensemble_methods: Optional[Dict[str, Type]] = None,
        feature_columns: Optional[List[str]] = None,
        text_column: str = "reviewText",
        n_estimators: int = 50,
        max_features: int = 50000,
        n_jobs: int = 1,
        random_state: int = RANDOM_SEED,
    ) -> None:
        """
        初始化CommentQualityPredictor对象

        参数:
                base_classifiers: 基分类器配置
                ensemble_methods: 集成方法配置
                feature_columns: 特征列
                text_column: 文本列名，特征列名称，默认为'reviewText'
                n_estimators: 基分类器数量，默认为50
                max_features: 文本向量化后的最大特征数，默认为10000
                n_jobs: 并行处理的线程数，默认为1
                random_state: 随机种子
        """

        # 设置默认基分类器
        self.base_classifiers = base_classifiers or {
            "SVM": BaseClassifierWrapper(
                LinearSVC(
                    C=0.1,
                    penalty="l2",
                    loss="squared_hinge",
                    dual=False,
                    max_iter=1000,
                    tol=0.01,
                    random_state=random_state,
                ),
                supports_weights=True,
            ),
            "DecisionTree": BaseClassifierWrapper(
                DecisionTreeClassifier(
                    max_depth=5,
                    min_samples_split=2,
                    min_samples_leaf=1,
                    random_state=random_state,
                ),
                supports_weights=True,
            ),
            # kNN不支持样本权重，设置为False
            # 'kNN': BaseClassifierWrapper(
            #     KNeighborsClassifier(
            #         n_neighbors = 5,
            #         weights = 'distance',
            #         algorithm = 'auto')),
            # 'NaiveBayes': BaseClassifierWrapper(GaussianNB(var_smoothing = 1e-10))
        }
        logger.info(f"配置基分类器: {', '.join(self.base_classifiers.keys())}")

        # 设置默认集成方法
        self.ensemble_methods = ensemble_methods or {
            "Bagging": BaggingClassifier,
            "AdaBoostM1": AdaBoostM1Classifier,
        }
        logger.info(f"配置集成方法: {', '.join(self.ensemble_methods.keys())}")

        # 设置默认特征列
        self.feature_columns = feature_columns or ["overall"]
        self.text_column = text_column
        self.random_state = random_state
        self.results = {}  # 存储模型评估结果
        self.feature_importance = {}  # 存储特征分析结果
        self.param_results = {}  # 存储参数分析结果
        self.n_estimators = n_estimators
        self.max_features = max_features
        self.n_jobs = n_jobs
        logger.info("评论质量预测器初始化完成")

    def train_and_evaluate(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_test: np.ndarray,
        y_test: np.ndarray,
    ) -> Dict[str, Dict[str, float]]:
        """
        训练并评估所有模型组合

        步骤:
        1. 遍历所有基分类器和集成方法
        2. 训练每个组合的模型
        3. 计算每个模型的AUC
        4. 记录结果

        参数:
                X_train: 训练特征
                y_train: 训练标签
                X_test: 测试特征
                y_test: 测试标签

        返回:
                Dict: 包含所有模型评估结果的字典
        """

        start_time = time.time()
        logger.info("开始训练和评估所有模型组合...")

        results = {}
        count = 0
        combinations = len(self.base_classifiers) * len(self.ensemble_methods)
        logger.info(f"将训练 {combinations} 种模型组合")

        for i, (base_name, base_clf) in enumerate(self.base_classifiers.items()):
            for ensemble_name, ensemble_cls in self.ensemble_methods.items():
                model_key = f"{ensemble_name}_{base_name}"
                logger.info(f"训练进度 {count + 1}/{combinations}: {model_key}")
                iter_start = time.time()
                count += 1

                try:
                    # 创建并训练集成分类器
                    logger.info(f"创建{ensemble_name}集成, 基分类器={base_name}")

                    ensemble_params = {
                        "base_estimator": base_clf,
                        "n_estimators": self.n_estimators,
                        "random_state": self.random_state,
                    }

                    if ensemble_name == "Bagging":
                        ensemble_params["n_jobs"] = self.n_jobs

                    # 创建集成模型
                    ensemble = ensemble_cls(**ensemble_params)

                    logger.info(f"开始训练{model_key}")
                    ensemble.fit(X_train, y_train)

                    # 预测并计算AUC
                    logger.info(f"评估{model_key}模型性能")
                    y_pred_proba = ensemble.predict_proba(X_test)[:, 1]
                    auc = roc_auc_score(y_test, y_pred_proba)
                    logger.info(f"{model_key} AUC = {auc:.4f}")

                    # 记录结果
                    iter_time = time.time() - iter_start
                    results[model_key] = {"auc": auc, "time": iter_time}

                except Exception as e:
                    logger.error(f"训练/评估 {model_key} 出错: {str(e)}")
                    results[model_key] = {"auc": 0.0, "time": 0.0}
                    continue

        self.results = results
        total_time = time.time() - start_time
        logger.info(f"所有模型训练和评估完成! 总耗时: {total_time:.2f}秒")
        return results

    def analyze_feature_impact(
        self,
        X_train_df: pd.DataFrame,
        y_train: np.ndarray,
        X_test_df: pd.DataFrame,
        y_test: np.ndarray,
        base_classifier: str = "DecisionTree",
    ) -> Dict[str, float]:
        """
        分析特征数量对模型性能的影响

        步骤:
        1. 使用文本特征训练模型
        2. 比较各组合的AUC

        参数:
                X_train_df: 训练特征DataFrame
                y_train: 训练标签
                X_test_df: 测试特征
                y_test: 测试标签
                base_classifier: 使用的基分类器名称

        返回:
                Dict: 包含不同特征组合结果的字典
        """

        start_time = time.time()
        logger.info(f"开始分析 (使用{base_classifier})...")

        if base_classifier not in self.base_classifiers:
            logger.warning(
                f"指定的基分类器{base_classifier}不存在，默认使用DecisionTree"
            )
            base_classifier = "DecisionTree"

        base_clf = self.base_classifiers[base_classifier]
        feature_results = {}

        logger.info("分析不同文本特征数量的影响...")
        text_feature_sizes = [
            1000,
            2000,
            3000,
            4000,
            5000,
            6000,
            7000,
            8000,
            9000,
            10000,
        ]

        # 只预处理文本一次，避免重复处理
        logger.info("预处理文本数据...")
        text_data_train = FeatureEngineer.preprocess_text_only(
            X_train_df[self.text_column]
        )
        text_data_test = FeatureEngineer.preprocess_text_only(
            X_test_df[self.text_column]
        )

        # 并行处理不同特征大小的实验
        def train_and_evaluate(size):
            logger.info(f"处理特征数量: {size}")
            try:
                # 创建向量化器
                vectorizer = TfidfVectorizer(
                    max_features=size,
                    ngram_range=(1, 3),
                    min_df=3,
                    max_df=0.95,
                    stop_words="english",
                )

                # 向量化文本数据
                X_train_text = vectorizer.fit_transform(text_data_train)
                X_test_text = vectorizer.transform(text_data_test)

                # 创建集成模型
                ensemble = BaggingClassifier(
                    base_estimator=base_clf,
                    n_estimators=self.n_estimators,
                    n_jobs=self.n_jobs,
                    random_state=self.random_state,
                )

                # 训练模型
                ensemble.fit(X_train_text, y_train)

                # 预测和评估
                y_pred_proba = ensemble.predict_proba(X_test_text)[:, 1]
                auc = roc_auc_score(y_test, y_pred_proba)
                logger.info(f"特征数量={size} AUC: {auc:.4f}")

                return size, auc

            except Exception as e:
                logger.error(f"处理特征数量 {size} 时出错: {str(e)}")
                return size, 0.0

        # 并行执行
        results = Parallel(n_jobs=self.n_jobs)(
            delayed(train_and_evaluate)(size) for size in text_feature_sizes
        )

        # 收集结果
        for size, auc in results:
            feature_results[f"text_{size}"] = auc

        self.feature_importance = feature_results

        # 绘制特征数量影响图
        self.plot_feature_impact()

        total_time = time.time() - start_time
        logger.info(f"文本特征数量影响分析完成! 总耗时: {total_time:.2f}秒")
        return feature_results

    def plot_feature_impact(self):
        """
        绘制特征数量对模型性能的影响图
        """

        if not self.feature_importance:
            logger.warning("没有特征重要性数据可绘制")
            return

        sizes = sorted([int(k.split("_")[1]) for k in self.feature_importance.keys()])
        aucs = [self.feature_importance[f"text_{size}"] for size in sizes]

        plt.figure(figsize=(12, 8))
        plt.plot(sizes, aucs, "o-", color="royalblue", linewidth=2, markersize=8)
        plt.title("文本特征数量对模型性能的影响", fontsize=14)
        plt.xlabel("特征数量", fontsize=12)
        plt.ylabel("AUC", fontsize=12)
        plt.grid(True, linestyle="--", alpha=0.7)
        plt.xticks(sizes, rotation=45)

        # 标记最高AUC点
        max_auc_idx = np.argmax(aucs)
        plt.annotate(
            f"Max AUC: {aucs[max_auc_idx]:.4f}",
            xy=(sizes[max_auc_idx], aucs[max_auc_idx]),
            xytext=(sizes[max_auc_idx] + 500, aucs[max_auc_idx] - 0.01),
            arrowprops=dict(facecolor="red", shrink=0.05),
        )

        plt.tight_layout()
        plt.show()

    def analyze_parameter_impact(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_test: np.ndarray,
        y_test: np.ndarray,
        base_classifier_name: str = "DecisionTree",
        ensemble_method: str = "Bagging",
    ) -> Dict[str, Dict[float, float]]:
        """
        分析参数对模型性能的影响

        步骤:
        1. 分析基分类器数量的影响
        2. 分析其他参数的影响（取决于集成方法）

        参数:
                X_train: 训练特征
                y_train: 训练标签
                X_test: 测试特征
                y_test: 测试标签
                base_classifier_name: 使用的基分类器名称
                ensemble_method: 使用的集成方法名称

        返回:
                Dict: 包含不同参数设置结果的字典
        """

        start_time = time.time()
        logger.info(
            f"开始参数影响分析 (使用{ensemble_method}_{base_classifier_name})..."
        )

        if base_classifier_name not in self.base_classifiers:
            logger.warning(
                f"指定的基分类器{base_classifier_name}不存在，使用DecisionTree"
            )
            base_classifier_name = "DecisionTree"

        base_clf = self.base_classifiers[base_classifier_name]
        param_results = {}

        # 1.分析基分类器数量的影响
        logger.info("分析1: 基分类器数量的影响")
        n_estimators_range = [5, 10, 20, 50, 100, 200, 300, 400, 500]
        n_estimators_results = {}

        # 并行处理不同参数配置
        def train_and_evaluate(n_estimators):
            try:
                logger.info(f"基分类器数量: {n_estimators}")

                # 创建集成模型
                if ensemble_method == "Bagging":
                    ensemble = BaggingClassifier(
                        base_estimator=base_clf,
                        n_estimators=n_estimators,
                        n_jobs=self.n_jobs,
                        random_state=self.random_state,
                    )
                else:  # AdaBoostM1
                    ensemble = AdaBoostM1Classifier(
                        base_estimator=base_clf,
                        n_estimators=n_estimators,
                        random_state=self.random_state,
                    )

                # 训练模型
                ensemble.fit(X_train, y_train)

                # 预测和评估
                y_pred_proba = ensemble.predict_proba(X_test)[:, 1]
                auc = roc_auc_score(y_test, y_pred_proba)
                logger.info(f"基分类器数量={n_estimators} AUC: {auc:.4f}")
                return n_estimators, auc
            except Exception as e:
                logger.error(f"处理基分类器数量 {n_estimators} 时出错: {str(e)}")
                return n_estimators, 0.0

        # 并行执行
        results = Parallel(n_jobs=self.n_jobs)(
            delayed(train_and_evaluate)(n) for n in n_estimators_range
        )

        # 收集结果
        for n, auc in results:
            n_estimators_results[n] = auc
        param_results["n_estimators"] = n_estimators_results

        # 2.分析Bagging的样本比例影响
        if ensemble_method == "Bagging":
            logger.info("分析2: Bagging样本比例的影响")
            max_samples_range = [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
            max_samples_results = {}

            def train_and_evaluate_sample(max_samples):
                try:
                    logger.info(f"样本比例: {max_samples}")
                    ensemble = BaggingClassifier(
                        base_estimator=base_clf,
                        n_estimators=self.n_estimators,
                        max_samples=max_samples,
                        n_jobs=self.n_jobs,
                        random_state=self.random_state,
                    )

                    ensemble.fit(X_train, y_train)
                    y_pred_proba = ensemble.predict_proba(X_test)[:, 1]
                    auc = roc_auc_score(y_test, y_pred_proba)
                    logger.info(f"样本比例={max_samples} AUC: {auc:.4f}")
                    return max_samples, auc

                except Exception as e:
                    logger.error(f"处理样本比例 {max_samples} 时出错: {str(e)}")
                    return max_samples, 0.0

            # 并行执行
            results = Parallel(n_jobs=self.n_jobs)(
                delayed(train_and_evaluate_sample)(m) for m in max_samples_range
            )

            # 收集结果
            for m, auc in results:
                max_samples_results[m] = auc
            param_results["max_samples"] = max_samples_results

        self.param_results = param_results

        # 绘制参数影响图
        self.plot_parameter_impact(ensemble_method)

        total_time = time.time() - start_time
        logger.info(f"参数影响分析完成! 总耗时: {total_time:.2f}秒")
        return param_results

    def plot_parameter_impact(self, ensemble_method: str):
        """
        绘制参数对模型性能的影响图

        参数:
                ensemble_method: 集成学习类型
        """

        if not self.param_results:
            logger.warning("没有参数结果数据可绘制")
            return

        plt.figure(figsize=(12, 8))

        # 绘制基分类器数量影响
        n_estimators = list(self.param_results["n_estimators"].keys())
        aucs = list(self.param_results["n_estimators"].values())

        plt.subplot(1, 2, 1)
        plt.plot(
            n_estimators, aucs, "s-", color="forestgreen", linewidth=2, markersize=8
        )
        plt.title("基分类器数量对性能的影响", fontsize=14)
        plt.xlabel("基分类器数量", fontsize=12)
        plt.ylabel("AUC", fontsize=12)
        plt.grid(True, linestyle="--", alpha=0.7)

        # 标记最高AUC点
        max_auc_idx = np.argmax(aucs)
        plt.annotate(
            f"Max AUC: {aucs[max_auc_idx]:.4f}",
            xy=(n_estimators[max_auc_idx], aucs[max_auc_idx]),
            xytext=(n_estimators[max_auc_idx] + 50, aucs[max_auc_idx] - 0.01),
            arrowprops=dict(facecolor="red", shrink=0.05),
        )

        # 如果是Bagging，绘制样本比例影响
        if ensemble_method == "Bagging" and "max_samples" in self.param_results:
            max_samples = list(self.param_results["max_samples"].keys())
            aucs = list(self.param_results["max_samples"].values())

            plt.subplot(1, 2, 2)
            plt.bar(range(len(max_samples)), aucs, color="coral", alpha=0.7)
            plt.xticks(range(len(max_samples)), [f"{m:.1f}" for m in max_samples])
            plt.title("样本比例对性能的影响", fontsize=14)
            plt.xlabel("样本比例", fontsize=12)
            plt.ylabel("AUC", fontsize=12)
            plt.grid(True, linestyle="--", alpha=0.7, axis="y")

            # 标记最高AUC点
            max_auc_idx = np.argmax(aucs)
            plt.annotate(
                f"Max AUC: {aucs[max_auc_idx]:.4f}",
                xy=(max_auc_idx, aucs[max_auc_idx]),
                xytext=(max_auc_idx + 0.2, aucs[max_auc_idx] - 0.01),
                arrowprops=dict(facecolor="red", shrink=0.05),
            )

        plt.tight_layout()
        plt.show()

    def print_results(self) -> None:
        """
        打印模型评估结果
        """

        if not self.results:
            logger.warning("没有结果可打印，请先运行train_and_evaluate方法")
            return

        # 创建DataFrame
        results_df = pd.DataFrame(
            {
                "Model": list(self.results.keys()),
                "AUC": [metrics["auc"] for metrics in self.results.values()],
                "Training Time (s)": [
                    metrics["time"] for metrics in self.results.values()
                ],
            }
        )

        # 按AUC排序并添加排名
        results_df = results_df.sort_values("AUC", ascending=False)
        results_df.insert(0, "Rank", range(1, len(results_df) + 1))

        # 使用tabulate打印表格
        print("\n" + "=" * 70)
        print("模型评估结果".center(66))
        print("=" * 70)

        # 转换DataFrame为适合tabulate的格式
        table_data = results_df.values.tolist()
        headers = ["Rank", "Model", "AUC", "Training Time (s)"]
        print(
            tabulate(
                table_data,
                headers=headers,
                floatfmt=(".0f", "", ".4f", ".2f"),
                tablefmt="grid",
                numalign="center",
                stralign="center",
            )
        )
        print("-" * 70)

        # 添加性能总结
        print(
            f"最佳模型: {results_df.iloc[0]['Model']} (AUC: {results_df.iloc[0]['AUC']:.4f})"
        )
        print(
            f"最差模型: {results_df.iloc[-1]['Model']} (AUC: {results_df.iloc[-1]['AUC']:.4f})"
        )
        print(f"总训练时间: {results_df['Training Time (s)'].sum():.2f} 秒")
        print("=" * 70)

    def print_feature_impact(self) -> None:
        """
        打印文本特征数量分析结果
        """

        if not self.feature_importance:
            logger.warning(
                "没有文本特征数量分析结果可打印，请先运行analyze_feature_impact方法"
            )
            return

        # 创建DataFrame
        features_df = pd.DataFrame(
            {
                "Feature Size": [
                    int(k.split("_")[1]) for k in self.feature_importance.keys()
                ],
                "AUC": list(self.feature_importance.values()),
            }
        )

        # 按特征数量排序
        features_df = features_df.sort_values("Feature Size")

        # 使用tabulate打印精美表格
        print("\n" + "=" * 60)
        print("文本特征数量对模型性能的影响".center(56))
        print("=" * 60)

        # 转换DataFrame为适合tabulate的格式
        table_data = features_df.values.tolist()
        headers = ["Feature Size", "AUC"]
        print(
            tabulate(
                table_data,
                headers=headers,
                floatfmt=(".0f", ".4f"),
                tablefmt="grid",
                numalign="center",
                stralign="center",
            )
        )
        print("-" * 60)

        # 添加分析总结
        max_idx = features_df["AUC"].idxmax()
        min_idx = features_df["AUC"].idxmin()
        print(
            f"最佳特征数量: {features_df.loc[max_idx, 'Feature Size']} (AUC: {features_df.loc[max_idx, 'AUC']:.4f})"
        )
        print(
            f"最差特征数量: {features_df.loc[min_idx, 'Feature Size']} (AUC: {features_df.loc[min_idx, 'AUC']:.4f})"
        )
        print(f"AUC变化范围: {features_df['AUC'].max() - features_df['AUC'].min():.4f}")
        print("=" * 60)

    def print_parameter_impact(self) -> None:
        """
        打印参数影响分析结果
        """

        if not self.param_results:
            logger.warning(
                "没有参数影响分析结果可打印，请先运行analyze_parameter_impact方法"
            )
            return

        # 打印参数表格
        print("\n" + "=" * 70)
        print("参数影响分析结果".center(66))
        print("=" * 70)

        # 处理基分类器数量结果
        n_estimators_results = self.param_results["n_estimators"]
        n_estimators_list = sorted(n_estimators_results.items(), key=lambda x: x[0])

        # 使用tabulate打印基分类器数量影响表
        print("\n[基分类器数量影响]".center(66))
        print("-" * 70)
        table_data = [(n, auc) for n, auc in n_estimators_list]
        headers = ["Number of Estimators", "AUC"]
        print(
            tabulate(
                table_data,
                headers=headers,
                floatfmt=(".0f", ".4f"),
                tablefmt="grid",
                numalign="center",
                stralign="center",
            )
        )
        print("-" * 70)

        # 添加分析总结
        max_auc_pair = max(n_estimators_list, key=lambda x: x[1])
        print(f"最佳基分类器数量: {max_auc_pair[0]}")
        print(f"最高AUC: {max_auc_pair[1]:.4f}")

        # 如果有样本比例结果
        if "max_samples" in self.param_results:
            max_samples_results = self.param_results["max_samples"]
            max_samples_list = sorted(max_samples_results.items(), key=lambda x: x[0])

            # 使用tabulate打印样本比例影响表
            print("\n\n[样本比例影响]".center(66))
            print("-" * 70)
            table_data = [(f"{ratio:.1f}", auc) for ratio, auc in max_samples_list]
            headers = ["Sample Ratio", "AUC"]
            print(
                tabulate(
                    table_data,
                    headers=headers,
                    floatfmt=("", ".4f"),
                    tablefmt="grid",
                    numalign="center",
                    stralign="center",
                )
            )
            print("-" * 70)

            # 添加分析总结
            max_auc_pair = max(max_samples_list, key=lambda x: x[1])
            print(f"最佳样本比例: {max_auc_pair[0]:.1f}")
            print(f"最高AUC: {max_auc_pair[1]:.4f}")

        print("=" * 70)

### 运行与结果展示

In [ ]:
# 数据路径
train_path = "data/train.csv"
test_path = "data/test.csv"
ground_truth_path = "groundTruth.csv"

In [ ]:
# 加载数据
data_loader = DataLoader(train_path, test_path, ground_truth_path)  # 初始化数据加载器
X_train_df, y_train, X_test_df, y_test = data_loader.load_data()

In [ ]:
# 特征工程（使用NLTK进行文本预处理）
feature_engineer = FeatureEngineer()  # 初始化特征工程（使用NLTK）
X_train = feature_engineer.fit_transform(X_train_df)  # 处理训练数据特征（使用NLTK）
X_test = feature_engineer.transform(X_test_df)  # 处理测试数据特征（使用NLTK）

In [ ]:
# 初始化预测器
predictor = CommentQualityPredictor(random_state=RANDOM_SEED)  # 初始化评论质量预测器

In [ ]:
# 训练和评估模型
results = predictor.train_and_evaluate(
    X_train, y_train, X_test, y_test
)  # 开始训练和评估模型

In [ ]:
# 分析文本特征数量影响
# feature_results = predictor.analyze_feature_impact(X_train_df, y_train, X_test_df, y_test) # 开始分析文本特征数量影响

In [ ]:
# 分析参数影响
# param_results = predictor.analyze_parameter_impact(X_train, y_train, X_test, y_test)

In [ ]:
# 打印结果
predictor.print_results()
# predictor.print_feature_impact()
# predictor.print_parameter_impact()